# 260421 Tool Calling + Agent 개념

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w7_tool_calling/llm_260421_tool_calling_intro.ipynb)

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-openai openai

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 강의 메모: Tool Calling 핵심

- **왜 필요한가**: 모델 고유 성능만으로 안 되는 문제(예: "strawberry의 r 개수", 실시간 날씨/환율) → 외부 함수 호출로 보완.
- **RAG 비유**: RAG가 청크를 검색해 컨텍스트로 답을 만들었다면, Tool Calling은 미리 정의된 **함수**를 호출해 그 결과로 답을 만든다. (답하기 전에 외부 자원을 가져온다는 흐름이 동일)
- **흐름**: 사용자 질문 → LLM이 bind된 툴 목록을 보고 "툴 필요 여부" 판단 → 필요하면 `tool_calls`(name + args) 반환 → 개발자가 실제 함수 실행 → 결과를 다시 LLM에 넘겨 최종 답변 생성.
- **토큰 비용 주의**: 툴 판단 + 파라미터 추출 + 결과 반영으로 호출이 2~3번 들어감. 에이전트가 토큰 많이 먹는 이유.

### 단계별로 다룰 것
1. raw dict로 툴 흉내내기 (개념 확인)
2. `@tool` 데코레이터 + `bind_tools` + `invoke` → `tool_calls` 확인
3. `tool_map`으로 name→함수 매핑해서 실제 실행
4. `StructuredTool.from_function` + Pydantic 스키마 (복잡한 입력 검증)
5. 여러 툴 동시 바인딩 / 멀티 호출
6. `ToolMessage`로 결과 회신해서 최종 답변까지 받기
7. `max_turns` 방어 로직 (무한 루프/토큰 낭비 방지)
8. `with_structured_output`으로 답변 포맷 강제
9. `tool_choice`로 특정 툴 강제 호출

> 비유: LLM은 "비서", 툴은 "비서가 들고 있는 도구함". 비서가 직접 모르는 건 도구함에서 꺼내 쓰고, 결과만 정리해서 보고한다.

## 강의 메모: 실무 팁 (트레이드오프 & 방어 로직)

- **트레이드오프**: 새 기술을 넣는다고 무조건 좋아지지 않음. 데이터/태스크에 따라 어떤 건 잘 되고 어떤 건 안 됨. 항상 "이게 우리 케이스에 맞나?" 점검.
- **답변 불안정성**: 같은 "서울 날씨 어때"도 어떨 땐 잘 나오고 어떨 땐 안 나옴 (모델 탐). → 영어/한글 차이로도 결과가 달라짐.
- **무한 루프 주의**: 툴이 잘못 매칭되면 LLM이 계속 다른 툴을 시도하며 토큰만 갉아먹음. → `max_turns`로 최대 턴 수 제한.
- **방어 로직 패턴**:
  - 툴이 우리가 정의한 게 아닌 다른 이름으로 오면(할루시네이션) "도구 없음" 처리
  - 툴 결과를 자연어 그대로 두지 말고 `with_structured_output` + Pydantic으로 포맷 강제 → 검증/파싱 쉬움
  - `tool_choice`로 특정 툴을 강제 사용시키면 안녕/잡담 같은 무관 입력에도 해당 툴이 무조건 호출됨 (의도된 강제는 OK, 아니면 부작용 주의)
- **Pydantic 스키마 디폴트값 함정**: `EtfSearchInput`의 `min_return=0.0`, `max_expense=1.0`처럼 디폴트를 잘못 잡으면 함수 필터에서 모두 걸러져 "결과 없음"이 나올 수 있음 → 데이터 분포에 맞게 디폴트 설정.